# Backward Model Hyperparameter Tuning

Tunes GMM (n_components, covariance_type) using BIC **separately for wrought and cast** composition spaces. Saves best params to `backward.wrought.GMM` and `backward.cast.GMM` in `hyperparams_config.json`.

In [1]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams

INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']

In [2]:
# Load composition data for wrought and cast (composition columns only)
WROUGHT_PATH = 'wrought_alloys_final.csv'
CAST_PATH = 'cleaned_cast_dataset.csv'

def load_wrought_compositions(path):
    if not os.path.exists(path):
        return None
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                if f.read(2) == b'PK':
                    df = pd.read_excel(path)
                else:
                    try:
                        df = pd.read_csv(path, encoding='utf-8')
                    except UnicodeDecodeError:
                        df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    for c in INPUT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df[INPUT_COLS].dropna()

def load_cast_compositions(path):
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')
    def extract(text):
        if pd.isna(text): return {}
        text = str(text).replace('AI', 'Al')
        m = re.search(r'\((.*?)\)', text)
        if not m: return {}
        f = m.group(1)
        comp = {}
        for el in INPUT_COLS:
            if el == 'Al': continue
            p = re.compile(rf"{el}(\d*\.?\d*)")
            h = p.search(f)
            if h:
                v = h.group(1)
                comp[el] = float(v) if (v and v != '.') else 0.5
        t = sum(comp.values())
        comp['Al'] = 100 - t if 0 < t < 100 else 0.0
        return comp
    ext = df['Alloy Name'].apply(extract)
    comp_df = pd.DataFrame(list(ext)).fillna(0.0)
    for c in INPUT_COLS:
        if c not in comp_df.columns: comp_df[c] = 0.0
    comp_df = comp_df[comp_df['Al'] > 1.0].reset_index(drop=True)
    return comp_df[INPUT_COLS].dropna()

X_wrought = load_wrought_compositions(WROUGHT_PATH)
X_cast = load_cast_compositions(CAST_PATH)
if X_wrought is not None:
    print(f'Wrought composition data: {X_wrought.shape}')
else:
    print('Wrought data not found.')
if X_cast is not None:
    print(f'Cast composition data: {X_cast.shape}')
else:
    print('Cast data not found.')

Wrought composition data: (868, 12)
Cast composition data: (41, 12)


In [3]:
# Tune GMM per dataset: n_components and covariance_type using BIC
n_components_range = [6, 8, 10, 12, 15, 20]
covariance_types = ['full', 'tied']

def tune_gmm_bic(X, dataset_name):
    if X is None or len(X) < 20:
        return None
    best_bic = np.inf
    best_params = None
    results = []
    for n in n_components_range:
        for cov in covariance_types:
            try:
                gmm = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
                gmm.fit(X)
                bic = gmm.bic(X)
                results.append({'n_components': n, 'covariance_type': cov, 'bic': bic})
                if bic < best_bic:
                    best_bic = bic
                    best_params = {'n_components': n, 'covariance_type': cov, 'random_state': 42}
            except Exception as e:
                print(f'  Skip n={n} cov={cov}: {e}')
    print(f'{dataset_name} GMM BIC (top 3):')
    for r in sorted(results, key=lambda x: x['bic'])[:3]:
        print(f"  n={r['n_components']}, cov={r['covariance_type']}: BIC={r['bic']:.0f}")
    print(f'  Best: {best_params}\n')
    return best_params

best_wrought_gmm = tune_gmm_bic(X_wrought, 'Wrought')
best_cast_gmm = tune_gmm_bic(X_cast, 'Cast')

Wrought GMM BIC (top 3):
  n=20, cov=full: BIC=-56134
  n=15, cov=full: BIC=-47276
  n=12, cov=full: BIC=-42195
  Best: {'n_components': 20, 'covariance_type': 'full', 'random_state': 42}



Cast GMM BIC (top 3):
  n=6, cov=full: BIC=-2041
  n=6, cov=tied: BIC=-1702
  n=8, cov=tied: BIC=-1606
  Best: {'n_components': 6, 'covariance_type': 'full', 'random_state': 42}



In [4]:
# Save backward GMM params (wrought and cast separately; merge into config)
if best_wrought_gmm is not None:
    save_hyperparams({'backward': {'wrought': {'GMM': best_wrought_gmm}}})
    print('Saved backward.wrought.GMM to hyperparams_config.json')
else:
    default_gmm = get_default_hyperparams('GMM')
    save_hyperparams({'backward': {'wrought': {'GMM': default_gmm}}})
    print('No wrought data; saved default backward.wrought.GMM')

if best_cast_gmm is not None:
    save_hyperparams({'backward': {'cast': {'GMM': best_cast_gmm}}})
    print('Saved backward.cast.GMM to hyperparams_config.json')
else:
    default_gmm = get_default_hyperparams('GMM')
    save_hyperparams({'backward': {'cast': {'GMM': default_gmm}}})
    print('No cast data; saved default backward.cast.GMM')

# Forward per-target params are used by 05_backward_wrought / 05_backward_cast and synthetic generators
by_target_w = load_hyperparams('wrought') and load_hyperparams('wrought').get('by_target')
if by_target_w:
    print('Forward wrought.by_target available for backward/synthetic pipelines.')
else:
    print('Run 01_hyperparameter_tuning_forward.ipynb first for wrought.by_target.')

Saved backward.wrought.GMM to hyperparams_config.json
Saved backward.cast.GMM to hyperparams_config.json
Forward wrought.by_target available for backward/synthetic pipelines.
